[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C70_RAG_Production_Course/03_query_side/03_query_side.ipynb)

# 03 · 查询侧改写与路由（路由 / 改写 / HyDE / 融合 / 过滤 / 否定 / 缓存 / 评测）

目标：把查询侧的四个动作变成**八个可测量的结论**。

本 notebook 你会亲手实现：
1. **路由** —— 无关上下文对答案的实际伤害，以及两类错误的不对称代价
2. **三种改写** —— 同义词表 / LLM 改写替身 / 语义漂移的事故
3. **HyDE** —— 伪答案**不需要正确**，只需要像一段文档
4. **多查询融合** —— 它降的是方差不是均值，用 RRF 量出来
5. **过滤前置 vs 后置** —— $(1-\phi)^k$ 的空结果概率曲线
6. **否定与数值约束** —— 向量检索对它们几乎失明，必须走结构化
7. **查询侧缓存键** —— 漏一项就会读到旧结果
8. **评测集自检** —— 用文档原句当查询会让词汇鸿沟这个失败消失

> 心智模型：**查询是流水线里唯一能在运行时改写、按请求生效、十分钟内回滚的东西。
> 所以它的实验成本最低，值得优先投入。**

## 0 · 环境与语料

In [ ]:
import os, re, json, math, hashlib
from collections import Counter, defaultdict

import numpy as np

DIM = 4096

def tokenize(text):
    text = text.lower()
    toks = re.findall(r'[a-z0-9]+', text)
    for run in re.findall(r'[\u4e00-\u9fff]+', text):
        toks += list(run)
        toks += [run[i:i + 2] for i in range(len(run) - 1)]
    return toks

def embed(text, dim=DIM):
    v = np.zeros(dim)
    for tok in tokenize(text):
        v[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def cos(a, b):
    return float(np.dot(a, b))

# 块 = (chunk_id, text, metadata)。metadata 是模块 01 抽出来的东西。
CHUNKS = [
    ('c01', '餐饮报销的单次上限是 200 元。',
     dict(section='报销', tenant='acme', year=2024, dept='all')),
    ('c02', '差旅报销的单次上限是 3000 元。',
     dict(section='报销', tenant='acme', year=2024, dept='all')),
    ('c03', '所有报销需在费用发生后 30 天内提交。',
     dict(section='报销', tenant='acme', year=2024, dept='all')),
    ('c04', '餐饮报销的单次上限是 150 元。',
     dict(section='报销', tenant='acme', year=2022, dept='all')),      # 旧版本
    ('c05', '正式员工每年享有 15 天带薪年假。',
     dict(section='休假', tenant='acme', year=2024, dept='all')),
    ('c06', '试用期员工每年享有 5 天带薪年假。',
     dict(section='休假', tenant='acme', year=2024, dept='all')),
    ('c07', '笔记本电脑的更换周期是 36 个月。',
     dict(section='设备', tenant='acme', year=2024, dept='it')),
    ('c08', '密码长度不得少于 12 位。',
     dict(section='安全', tenant='acme', year=2024, dept='it')),
    ('c09', '单笔金额超过 50 万元的合同需总经理签批。',
     dict(section='合同', tenant='acme', year=2024, dept='legal')),
    ('c10', '合同归档保存 10 年。',
     dict(section='合同', tenant='acme', year=2024, dept='legal')),
    ('c11', '外发文件必须经过审批。',
     dict(section='安全', tenant='acme', year=2024, dept='all')),
    ('c12', '每位员工每年有 2000 元培训预算。',
     dict(section='培训', tenant='acme', year=2024, dept='all')),
    # 另一个租户 —— 第 5 节的多租户过滤要用
    ('x01', '餐饮报销的单次上限是 80 元。',
     dict(section='报销', tenant='globex', year=2024, dept='all')),
    ('x02', '正式员工每年享有 10 天带薪年假。',
     dict(section='休假', tenant='globex', year=2024, dept='all')),
]

MAT = np.stack([embed(t) for _, t, _ in CHUNKS])
ID2I = {cid: i for i, (cid, _, _) in enumerate(CHUNKS)}

def retrieve(query, k=3, candidates=None):
    """candidates: chunk_id 的集合；None 表示全库。返回 [(chunk_id, score)]。"""
    idx = ([ID2I[c] for c in candidates] if candidates is not None
           else list(range(len(CHUNKS))))
    if not idx:
        return []
    s = MAT[idx] @ embed(query)
    order = np.argsort(-s)[:k]
    return [(CHUNKS[idx[o]][0], float(s[o])) for o in order]

print(f'{len(CHUNKS)} 个块 · {len({m["tenant"] for _, _, m in CHUNKS})} 个租户')
print('检索「餐饮报销上限」:', retrieve('餐饮报销的上限是多少', k=3))

## 1 · 路由：无关上下文的实际伤害

回答器是一个规则替身：**从上下文里取第一个数字**。
这是「模型被上下文带偏」这个真实行为的一个可控、可观测的模型。

In [ ]:
def answer_first_number(context):
    m = re.search(r'\d+', context)
    return m.group(0) if m else None

# 三个不需要检索的查询 —— 答案与语料无关
NO_RETRIEVAL = {
    '3000 除以 12 等于多少': '250',
    '一年有多少个月':         '12',
    '一天有多少小时':         '24',
}

def answer_without_retrieval(q):
    """不检索：直接算（这里用查表替身）。"""
    return NO_RETRIEVAL[q]

def answer_with_retrieval(q, k=3):
    """检索后把 top-k 拼成上下文，再取第一个数字。"""
    hits = retrieve(q, k=k)
    ctx = ''.join(dict(zip([c for c, _, _ in CHUNKS],
                          [t for _, t, _ in CHUNKS]))[cid] for cid, _ in hits)
    return answer_first_number(ctx)

print(f"{'查询':<20}{'不检索':>8}{'检索后':>8}{'正确答案':>10}")
n_ok_no, n_ok_yes = 0, 0
for q, gold in NO_RETRIEVAL.items():
    a_no = answer_without_retrieval(q)
    a_yes = answer_with_retrieval(q)
    n_ok_no += (a_no == gold); n_ok_yes += (a_yes == gold)
    print(f'{q:<20}{str(a_no):>8}{str(a_yes):>8}{gold:>10}')

print(f'\n不检索正确率 {n_ok_no / len(NO_RETRIEVAL):.0%} | '
      f'一律检索正确率 {n_ok_yes / len(NO_RETRIEVAL):.0%}')
assert n_ok_no == len(NO_RETRIEVAL), '这些查询本来就答得对'
assert n_ok_yes < n_ok_no, '对不该检索的查询做检索会把答案带偏'
print('✅ 「一律检索」这个默认值对这类查询是净损害——它注入了具体但无关的数字。')

In [ ]:
# --- 路由器：规则 + 兜底，且刻意偏向检索 ---
SKIP_PATTERNS = [
    r'^\s*(你好|您好|hi|hello)\s*[。！!？?]*\s*$',      # 闲聊
    r'[\d]+\s*(加|减|乘|除以|\+|-|\*|/)\s*[\d]+',      # 算术
    r'^(一年有多少个月|一天有多少小时)$',                  # 明确的常识
]

def route(query):
    """返回 'skip' 或 'retrieve'。偏向 retrieve —— 漏检索比误检索更贵。"""
    for pat in SKIP_PATTERNS:
        if re.search(pat, query):
            return 'skip'
    return 'retrieve'

NEED_RETRIEVAL = ['餐饮报销的单次上限是多少', '正式员工年假多少天',
                  '密码最短多少位', '合同归档保存多久']
cases = [(q, 'skip') for q in NO_RETRIEVAL] + [(q, 'retrieve') for q in NEED_RETRIEVAL]
cases.append(('你好', 'skip'))

wrong_skip = wrong_retrieve = 0
for q, want in cases:
    got = route(q)
    if got != want:
        (globals().__setitem__('wrong_skip', wrong_skip + 1) if want == 'retrieve'
         else globals().__setitem__('wrong_retrieve', wrong_retrieve + 1))
    print(f'{q:<22} 期望 {want:<9} 实际 {got:<9} {"✓" if got == want else "✗"}')

miss = sum(1 for q, want in cases if want == 'retrieve' and route(q) == 'skip')
over = sum(1 for q, want in cases if want == 'skip' and route(q) == 'retrieve')
print(f'\n漏检索（该查没查）{miss} 个 | 误检索（不该查却查了）{over} 个')
assert miss == 0, '漏检索的代价更高，路由器必须先保证这一项为 0'
print('✅ 规则路由器的正确设计目标是「漏检索为 0」，而不是「总准确率最高」。')
print('   两类错误的代价不对称：漏检索 → 答不上来；误检索 → 注入噪声。')
print('   所以先用规则覆盖明确的跳过模式，其余全部检索——')
print('   不要一上来就用 LLM 分类器，它的误检索率你没有数据去估计。')

## 2 · 改写（一）：同义词表与语义漂移事故

同义词表便宜、可审计、可回滚到具体一行。
但**任何改写都可能改变语义**——这一节把那个事故做出来。

In [ ]:
SYNONYM = {'吃饭': '餐饮', '最多能花': '单次上限是', '休假': '年假',
           '出差': '差旅', '电脑': '笔记本电脑'}

def rewrite_synonym(q, table=SYNONYM):
    out = q
    for user_word, doc_word in table.items():
        out = out.replace(user_word, doc_word)
    return out

GAP_CASES = {
    '吃饭最多能花多少钱':   'c01',
    '出差最多能花多少钱':   'c02',
    '电脑多久换一次':       'c07',
}
print(f"{'原查询':<20}{'改写后':<26}{'原 top1':>10}{'改写 top1':>11}")
before = after = 0
for q, gold in GAP_CASES.items():
    q2 = rewrite_synonym(q)
    t1 = retrieve(q, k=1)[0][0]
    t2 = retrieve(q2, k=1)[0][0]
    before += (t1 == gold); after += (t2 == gold)
    print(f'{q:<20}{q2:<26}{t1:>10}{t2:>11}')
print(f'\ntop1 命中: 改写前 {before}/{len(GAP_CASES)} → 改写后 {after}/{len(GAP_CASES)}')
assert after > before, '同义词改写应当提升命中'

# --- 语义漂移事故 ---
DRIFT_TABLE = {'不含增值税': '增值税'}      # 一个真实会发生的坏映射
bad_q = '不含增值税的发票怎么处理'
print(f'\n语义漂移: 「{bad_q}」 → 「{rewrite_synonym(bad_q, DRIFT_TABLE)}」')
print('   「不含」被吃掉了——改写把否定丢了，语义反转。')
assert '不含' not in rewrite_synonym(bad_q, DRIFT_TABLE)
print('✅ 所以纪律是：**改写永远不替换原查询，而是作为并行的一路**（第 4 节融合）。')

## 3 · 改写（二）：HyDE —— 伪答案不需要正确

HyDE 换的不是「问对问题」，而是**换一个更接近文档分布的嵌入输入**。
关键实验：一个**内容并不正确**的粗糙伪答案，也能把相似度拉起来。

In [ ]:
TARGET = '餐饮报销的单次上限是 200 元。'
Q = '吃饭最多能花多少钱'

VARIANTS = [
    ('原查询（问句）',       Q),
    ('同义词改写（仍是问句）', rewrite_synonym(Q)),
    ('粗糙伪答案（内容错）',  '餐饮报销的单次上限是若干元。'),
    ('更差的伪答案（数字错）', '餐饮报销的单次上限是 500 元。'),
    ('理想伪答案（内容对）',  TARGET),
]
print(f"{'嵌入输入':<24}{'与目标块的相似度':>18}{'top1':>8}")
sims = {}
for label, text in VARIANTS:
    s = cos(embed(text), embed(TARGET))
    sims[label] = s
    print(f'{label:<24}{s:>18.3f}{retrieve(text, k=1)[0][0]:>8}')

s_q = sims['原查询（问句）']
s_rough = sims['粗糙伪答案（内容错）']
s_wrongnum = sims['更差的伪答案（数字错）']
assert s_rough > 5 * s_q, f'粗糙伪答案应当远好于原查询: {s_rough:.3f} vs {s_q:.3f}'
assert s_wrongnum > 5 * s_q, '连数字都写错的伪答案也一样有效'
print(f'\n✅ 原查询 {s_q:.3f} → 粗糙伪答案 {s_rough:.3f}（{s_rough / s_q:.0f} 倍）。')
print(f'   连数字写错的伪答案也有 {s_wrongnum:.3f}——**伪答案不需要正确**。')
print('   它只需要在正确的文体与词汇分布里（陈述句、有单位、用文档的词）。')
print()
print('   两个推论：')
print('   ① HyDE 的伪答案不必用高温度采样多个，也不必追求事实正确；')
print('   ② 但它会放大模型的知识偏见——伪答案里出现错误**实体**时')
print('      （不是错误数字），会把检索带到错误的地方。所以仍然要保留原查询那一路。')

## 4 · 多查询融合：它降的是方差

RRF 定义见 C11 模块 03，这里只用。
关键测量：**跨改写的召回率标准差**，单路 vs 融合。

In [ ]:
def rrf(rankings, k0=60, k=3):
    """rankings: [[chunk_id, ...], ...]。返回融合后的 top-k。"""
    score = defaultdict(float)
    for r in rankings:
        for rank, cid in enumerate(r, start=1):
            score[cid] += 1.0 / (k0 + rank)
    return [cid for cid, _ in sorted(score.items(), key=lambda kv: -kv[1])[:k]]

# 一组「好坏不一」的改写器 —— 真实系统里你事先不知道哪个好
REWRITERS = {
    '原查询':          lambda q: q,
    '同义词':          lambda q: rewrite_synonym(q),
    'HyDE(粗糙)':      lambda q: rewrite_synonym(q).replace('多少钱', '若干元。'),
    '坏改写(丢主题词)':  lambda q: re.sub(r'吃饭|出差|电脑|休假|密码|合同', '', q),
}
EVAL = {'吃饭最多能花多少钱': 'c01', '出差最多能花多少钱': 'c02',
        '电脑多久换一次': 'c07', '休假有几天': 'c05',
        '密码要多长': 'c08', '合同要存多久': 'c10'}

# 用 k=1 而不是 k=3：这个语料只有 14 个块，k=3 时几乎所有方案都饱和到 1.0，
# 看不出任何差别。**指标饱和时的对比是没有信息量的**（C66-01 的老问题）。
K_FUSE = 1

def recall_at_k(rewriter, k=K_FUSE):
    hit = 0
    for q, gold in EVAL.items():
        got = [cid for cid, _ in retrieve(rewriter(q), k=k)]
        hit += gold in got
    return hit / len(EVAL)

singles = {name: recall_at_k(fn) for name, fn in REWRITERS.items()}
for name, r in singles.items():
    print(f'{name:<18} recall@{K_FUSE} = {r:.2f}')

def recall_fused(k=K_FUSE):
    hit = 0
    for q, gold in EVAL.items():
        rankings = [[cid for cid, _ in retrieve(fn(q), k=k * 3)]
                    for fn in REWRITERS.values()]
        hit += gold in rrf(rankings, k=k)
    return hit / len(EVAL)

fused = recall_fused()
vals = list(singles.values())
print(f'\n单路: 均值 {np.mean(vals):.2f}  标准差 {np.std(vals):.3f}  '
      f'最好 {max(vals):.2f}  最差 {min(vals):.2f}')
print(f'融合: {fused:.2f}')

assert fused > min(vals), '融合必须明显好于最差的那一路'
assert fused >= np.mean(vals), '融合应当不低于单路均值'
assert fused < max(vals), '而且它**不保证**超过最好的那一路——本例就没有'
print(f'\n✅ 融合 {fused:.2f}：高于均值 {np.mean(vals):.2f}、'
      f'远高于最差 {min(vals):.2f}，但**低于最好的单路 {max(vals):.2f}**。')
print('   这就是融合的真实性质：它买的是「不会掉到最差」，不是「拿到最好」。')
print('   而你事先并不知道线上会抽到哪一路——如果知道，就不需要融合了。')
print('   这也解释了一个常见的失望：「加了融合，最好情况反而降了」。这是预期行为。')
print()
print('   实践中 m=3（原查询 + 同义改写 + HyDE）是常见甜点，')
print('   关键不是路数多，而是**这三路的失败模式互不相同**：')
print('   原查询败于词汇鸿沟，同义改写败于表外词，HyDE 败于错误实体。')

## 5 · 过滤：前置 vs 后置

后置过滤返回空的概率是 $(1-\phi)^k$。
$k=10, \phi=0.05$ 时是 **0.60**——六成请求返回空，而检索层不报错。

In [ ]:
def filter_ids(pred):
    return {cid for cid, _, m in CHUNKS if pred(m)}

def retrieve_prefilter(query, pred, k=3):
    return retrieve(query, k=k, candidates=filter_ids(pred))

def retrieve_postfilter(query, pred, k=3):
    hits = retrieve(query, k=k)                       # 先全库 top-k
    keep = filter_ids(pred)
    return [(cid, s) for cid, s in hits if cid in keep]

# 场景：这个用户只被授权看 legal 部门的文档（φ = 2/14）
IS_LEGAL = lambda m: m['dept'] == 'legal'
phi = len(filter_ids(IS_LEGAL)) / len(CHUNKS)
print(f'legal 部门的文档占全库 φ = {phi:.1%}')
Q5 = '餐饮报销的单次上限是多少'
print('全库 top3      :', [c for c, _ in retrieve(Q5, k=3)], '← 里面一条 legal 的都没有')
print('前置过滤 top3   :', retrieve_prefilter(Q5, IS_LEGAL, k=3))
print('后置过滤 top3   :', retrieve_postfilter(Q5, IS_LEGAL, k=3))
assert len(retrieve_prefilter(Q5, IS_LEGAL, k=3)) > 0, '前置过滤必须拿到结果'
assert len(retrieve_postfilter(Q5, IS_LEGAL, k=3)) == 0, '后置过滤在这里返回空'
print()
print('两者的区别不是「效果差一点」：')
print('  前置 → 「在你有权看的文档里，与这个问题最相关的是这两条」（哪怕相关性不高）')
print('  后置 → 「什么都没有」，而检索层不报错。')

# --- 空结果概率曲线 ---
print(f"\n{'φ':>7}{'k=5 空结果概率':>16}{'k=10':>10}{'k=20':>10}")
for ph in [0.5, 0.2, 0.1, 0.05, 0.01]:
    row = [(1 - ph) ** k for k in (5, 10, 20)]
    print(f'{ph:>7.2f}{row[0]:>16.3f}{row[1]:>10.3f}{row[2]:>10.3f}')
assert abs((1 - 0.05) ** 10 - 0.5987) < 1e-3
print('\n✅ φ=5%、k=10 时后置过滤有 60% 的请求返回空。')
print('   这个错误在测试库上常常不暴露——测试库里满足条件的占比很高。')
print('   上线后租户变多、时间窗变窄，φ 一掉，top-k 就被清空了。')

In [ ]:
# --- 两个必须报警的事件 ---
def retrieve_guarded(query, pred, k=3, min_candidates=3):
    cands = filter_ids(pred)
    alerts = []
    if len(cands) == 0:
        alerts.append('过滤后候选集为空——过滤条件很可能写错了')
    elif len(cands) < min_candidates:
        alerts.append(f'过滤后候选集只有 {len(cands)} 条（< {min_candidates}）')
    return retrieve(query, k=k, candidates=cands), alerts

# 事件一：条件写错 → 候选空
_, a1 = retrieve_guarded(Q5, lambda m: m['section'] == '不存在的章节')
# 事件二：tenant 是 None → 在「过滤」语义下会全不匹配（或更糟：全匹配）
_, a2 = retrieve_guarded(Q5, lambda m: m['tenant'] == None)
# 正常
_, a3 = retrieve_guarded(Q5, lambda m: m['tenant'] == 'acme')
for name, a in [('章节写错', a1), ('tenant=None', a2), ('正常', a3)]:
    print(f'{name:<14} {a if a else "无告警"}')
assert a1 and a2 and not a3
print('\n✅ 「过滤后候选集为空」必须报警而不是静默返回空。')
print('   多租户下这一条是**安全边界**：租户过滤必须前置、且必须是不可绕过的默认值。')
print('   靠调用方「记得传 tenant」的设计，把一次跨租户泄漏的距离缩短到一行漏写的代码。')

## 6 · 否定与数值约束：向量检索几乎失明

In [ ]:
NEG_PAIRS = [
    ('餐饮报销可以报销酒水',   '餐饮报销不可以报销酒水'),
    ('包含增值税的发票',       '不包含增值税的发票'),
    ('2024 年之后生效的条款',  '2024 年之前生效的条款'),
    ('超过 5000 元的合同',     '不超过 5000 元的合同'),
]
print(f"{'A':<24}{'B（语义相反）':<26}{'cos':>7}")
sims = []
for a, b in NEG_PAIRS:
    s = cos(embed(a), embed(b))
    sims.append(s)
    print(f'{a:<24}{b:<26}{s:>7.3f}')
assert min(sims) > 0.6, '语义相反的句子在向量空间里依然很近'
print(f'\n✅ 语义完全相反的句对，相似度 {min(sims):.2f}–{max(sims):.2f}。')
print('   一个「不」字改变了全部语义，却几乎不改变向量。')
print('   本课的玩具嵌入放大了这个问题，但真实句嵌入在否定上确实也弱——')
print('   这是一个被反复报告的已知局限，不是实现问题。')

In [ ]:
# --- 正确做法：把硬约束抽成过滤器 ---
def extract_constraints(query):
    """把查询里的硬约束抽出来，返回 (语义部分, 约束 dict)。"""
    cons, sem = {}, query
    m = re.search(r'(\d{4})\s*年(之后|以后|之前|以前)', query)
    if m:
        yr, direction = int(m.group(1)), m.group(2)
        cons['year'] = ('>=', yr) if direction in ('之后', '以后') else ('<=', yr)
        sem = sem.replace(m.group(0), '')
    m = re.search(r'(不?超过)\s*([\d.]+)\s*(万?)元', query)
    if m:
        val = float(m.group(2)) * (10000 if m.group(3) else 1)
        cons['amount'] = ('<=', val) if m.group(1).startswith('不') else ('>', val)
        sem = sem.replace(m.group(0), '')
    # 注意 \w 而不是只有中文：部门名可能是英文（it / legal），
    # 而「只匹配中文」这种偷懒的正则会让约束静默失效——过滤条件没生效比写错更难发现。
    m = re.search(r'只看\s*([\w\u4e00-\u9fff]+?)\s*(部门|的)', query)
    if m:
        cons['dept'] = ('==', m.group(1))
        sem = sem.replace(m.group(0), '')
    return sem.strip(), cons

for q in ['2024 年之后生效的报销规定', '超过 5 万元的合同怎么审批',
          '只看 it 部门的设备规定', '餐饮报销上限多少']:
    sem, cons = extract_constraints(q)
    print(f'{q:<24} → 语义「{sem}」 约束 {cons}')

def apply_constraints(cons):
    def pred(m):
        for field, (op, val) in cons.items():
            v = m.get(field)
            if v is None:
                return False
            if op == '>=' and not v >= val: return False
            if op == '<=' and not v <= val: return False
            if op == '>' and not v > val:  return False
            if op == '==' and not v == val: return False
        return True
    return pred

# 时效过滤把旧版本挡在外面（c04 是 2022 年的 150 元）
sem, cons = extract_constraints('2024 年之后生效的餐饮报销上限')
no_filter = [cid for cid, _ in retrieve('餐饮报销上限', k=3)]
with_filter = [cid for cid, _ in retrieve(sem or '餐饮报销上限', k=3,
                                          candidates=filter_ids(apply_constraints(cons)))]
print(f'\n不带时效过滤 top3: {no_filter}   ← c04 是 2022 年的旧值 150 元')
print(f'带时效过滤   top3: {with_filter}')
assert 'c04' in no_filter, '不过滤时旧版本会被召回'
assert 'c04' not in with_filter, '时效过滤必须把旧版本挡住'
print('✅ 否定、数值区间、时间窗、集合成员——这四类必须走结构化过滤。')
print('   而这依赖模块 01 把对应的元数据抽全：抽不到的字段，这里就没有可用的过滤器。')

## 7 · 查询侧缓存键

In [ ]:
QUERY_CACHE_FIELDS = ('query', 'route_version', 'rewriter_id', 'prompt_sha',
                      'synonym_version', 'model_id', 'temperature', 'm_expand')

def query_cache_key(cfg):
    norm = {k: cfg.get(k) for k in QUERY_CACHE_FIELDS}
    return hashlib.sha256(
        json.dumps(norm, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:12]

BASE = dict(query='吃饭最多能花多少钱', route_version='r3', rewriter_id='syn+hyde',
            prompt_sha='9af31c', synonym_version='2026-08-01',
            model_id='m-small', temperature=0.0, m_expand=3,
            log_level='INFO', request_id='req-771')

k0 = query_cache_key(BASE)
print('基线键:', k0)
for field, val in [('query', '出差最多能花多少钱'), ('route_version', 'r4'),
                   ('rewriter_id', 'syn'), ('prompt_sha', 'deadbe'),
                   ('synonym_version', '2026-09-01'), ('model_id', 'm-large'),
                   ('temperature', 0.7), ('m_expand', 5)]:
    c = dict(BASE); c[field] = val
    changed = query_cache_key(c) != k0
    print(f'  改 {field:<18} → 键变了 {changed}')
    assert changed, f'{field} 必须进缓存键'
for field, val in [('log_level', 'DEBUG'), ('request_id', 'req-999')]:
    c = dict(BASE); c[field] = val
    assert query_cache_key(c) == k0, f'{field} 不该进缓存键'
print('\n✅ 八个字段进键，与请求无关的两个不进。')
print('   最常被漏的是 prompt_sha——改了改写 prompt 而键不变，')
print('   症状是「我明明改了 prompt，效果一点没变」，而它会被误读成「这个改动没用」。')
print('   （与 C68 模块 02 的缓存键是同一条纪律，只是对象换成了查询处理配置。）')
print()
print('   还有一条：temperature > 0 时**不该缓存单次改写结果**——')
print('   要么把温度固定为 0，要么缓存「一整组」改写并整组复用。')

## 8 · 评测集自检：词重叠率

用文档原句改成的查询与目标块共享大量词，
于是**词汇鸿沟这个失败在你的评测集上根本不存在**。

In [ ]:
def word_overlap(query, chunk_text):
    a, b = set(tokenize(query)), set(tokenize(chunk_text))
    return len(a & b) / len(a) if a else 0.0

TEXT = {cid: t for cid, t, _ in CHUNKS}

# 两种构造方式
FROM_DOC = {'餐饮报销的单次上限是多少': 'c01', '正式员工每年享有多少天带薪年假': 'c05',
            '笔记本电脑的更换周期是多久': 'c07', '密码长度不得少于多少位': 'c08'}
FROM_USER = {'吃饭最多能花多少钱': 'c01', '休假有几天': 'c05',
             '电脑多久换一次': 'c07', '密码要多长': 'c08'}

K_EVAL = 1   # 同第 4 节：k=3 时两个集合都饱和到 1.00，看不出差别

def eval_set_report(qs, k=K_EVAL):
    ov = [word_overlap(q, TEXT[g]) for q, g in qs.items()]
    hit = sum(1 for q, g in qs.items() if g in [c for c, _ in retrieve(q, k=k)])
    return float(np.mean(ov)), hit / len(qs)

ov_doc, rec_doc = eval_set_report(FROM_DOC)
ov_user, rec_user = eval_set_report(FROM_USER)
print(f"{'评测集构造方式':<22}{'平均词重叠':>12}{'recall@' + str(K_EVAL):>11}")
print(f'{"用文档原句改成问题":<22}{ov_doc:>12.2f}{rec_doc:>11.2f}')
print(f'{"真实用户的问法":<22}{ov_user:>12.2f}{rec_user:>11.2f}')

assert ov_doc > 2 * ov_user, '文档原句构造的查询词重叠显著更高'
assert rec_doc > rec_user, '于是它测出的 recall 也虚高'

# 在两个集合上分别看「查询改写」的收益
def gain_of_rewrite(qs, k=K_EVAL):
    base = sum(1 for q, g in qs.items() if g in [c for c, _ in retrieve(q, k=k)])
    rew = sum(1 for q, g in qs.items()
              if g in [c for c, _ in retrieve(rewrite_synonym(q), k=k)])
    return (rew - base) / len(qs)

g_doc, g_user = gain_of_rewrite(FROM_DOC), gain_of_rewrite(FROM_USER)
print(f'\n查询改写的收益: 在「文档原句」集上 {g_doc:+.0%}，'
      f'在「真实问法」集上 {g_user:+.0%}')
assert g_user > g_doc, '在文档原句集上会得出「改写没收益」的错误结论'
print('✅ 这是本模块最重要的一条：')
print(f'   在文档原句构造的评测集上，查询改写的收益是 {g_doc:+.0%}——')
print('   于是你会得出「查询改写没用」这个结论。而它只是被评测集掩盖了。')
print('   便宜的自检：算评测查询与目标块的词重叠率，')
print('   显著高于真实流量的重叠率就说明评测集偏乐观了。')

## ✏️ 练习 1：完整的查询侧流水线

把四个动作串起来：`route` → `rewrite/expand` → `filter` → `retrieve` → `rrf`。

实现 `query_pipeline(query, k=3, m=3)`，返回
`dict(routed, queries, constraints, candidates_n, hits, alerts)`：
- `routed == 'skip'` 时**直接返回**，`hits` 为空、`queries` 只含原查询
- 约束抽取后，语义部分为空则用原查询
- **过滤必须前置**（作用在候选集上）
- `m` 路并行检索后用 RRF 融合
- 过滤后候选集为空要进 `alerts`

In [ ]:
def query_pipeline(query, k=3, m=3):
    """返回 dict(routed, queries, constraints, candidates_n, hits, alerts)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
# a) 不该检索的查询直接放行
r = query_pipeline('3000 除以 12 等于多少')
assert r['routed'] == 'skip' and r['hits'] == [] and len(r['queries']) == 1

# b) 词汇鸿沟的查询：融合后应当命中
r = query_pipeline('吃饭最多能花多少钱')
assert r['routed'] == 'retrieve'
assert len(r['queries']) >= 2, '至少要有原查询 + 一个改写'
assert r['queries'][0] == '吃饭最多能花多少钱', '第一路必须是原查询（兜底）'
assert 'c01' in r['hits'], r['hits']

# c) 带时效约束：旧版本必须被挡住，且过滤前置（候选数 < 全库）
r = query_pipeline('2024 年之后生效的餐饮报销上限')
assert r['constraints'].get('year') == ('>=', 2024), r['constraints']
assert r['candidates_n'] < len(CHUNKS), '过滤必须前置，候选集应当变小'
assert 'c04' not in r['hits'], '2022 年的旧值必须被挡住'
assert 'c01' in r['hits']

# d) 过滤条件写错 → 告警
r = query_pipeline('只看 nonexist 部门的设备规定')
assert r['alerts'], '候选集为空必须告警'
print('✅ 练习 1 通过：四个动作串起来，且过滤前置、原查询兜底')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def query_pipeline(query, k=3, m=3):
    alerts = []
    routed = route(query)
    if routed == 'skip':
        return dict(routed='skip', queries=[query], constraints={},
                    candidates_n=0, hits=[], alerts=alerts)
    sem, cons = extract_constraints(query)
    base = sem if sem else query
    # 扩展：原查询永远是第一路（兜底）
    variants = [query, rewrite_synonym(base)]
    variants.append(rewrite_synonym(base).replace('多少钱', '若干元。'))   # HyDE 替身
    variants = list(dict.fromkeys(variants))[:m]
    # 过滤前置
    cands = filter_ids(apply_constraints(cons)) if cons else None
    n_cand = len(cands) if cands is not None else len(CHUNKS)
    if cands is not None and len(cands) == 0:
        alerts.append('过滤后候选集为空——过滤条件很可能写错了')
        return dict(routed=routed, queries=variants, constraints=cons,
                    candidates_n=0, hits=[], alerts=alerts)
    rankings = [[cid for cid, _ in retrieve(v, k=k * 2, candidates=cands)]
                for v in variants]
    return dict(routed=routed, queries=variants, constraints=cons,
                candidates_n=n_cand, hits=rrf(rankings, k=k), alerts=alerts)

assert query_pipeline('3000 除以 12 等于多少')['routed'] == 'skip'
r = query_pipeline('吃饭最多能花多少钱')
assert r['queries'][0] == '吃饭最多能花多少钱' and 'c01' in r['hits']
r = query_pipeline('2024 年之后生效的餐饮报销上限')
assert r['candidates_n'] < len(CHUNKS) and 'c04' not in r['hits'] and 'c01' in r['hits']
assert query_pipeline('只看 nonexist 部门的设备规定')['alerts']
print('✅ 参考答案 1 通过')
print('   两个容易写错的地方：')
print('   ① 过滤要作为 candidates 传进检索，而不是在 hits 上筛（第 5 节的 (1-φ)^k）；')
print('   ② 原查询必须是第一路——改写坏掉时它是唯一的兜底。')

## ✏️ 练习 2：后置过滤的空结果概率

实现 `empty_prob(k, phi)` 与 `min_k_for_target(phi, target)`：
- `empty_prob(k, phi)` = 后置过滤返回空的概率 $(1-\phi)^k$
- `min_k_for_target(phi, target)` = 让空结果概率不超过 `target` 所需的最小 `k`

这个函数的用途：**如果你被迫用后置过滤（比如向量库不支持前置过滤），
它告诉你 k 要放大到多少才勉强可用。**

In [ ]:
def empty_prob(k, phi):
    """后置过滤返回空的概率。"""
    # TODO
    raise NotImplementedError

def min_k_for_target(phi, target):
    """最小的 k 使 empty_prob(k, phi) <= target。phi<=0 时返回 None。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
assert abs(empty_prob(10, 0.05) - 0.5987) < 1e-3
assert empty_prob(0, 0.3) == 1.0
assert empty_prob(5, 1.0) == 0.0

assert min_k_for_target(0.5, 0.01) == 7, min_k_for_target(0.5, 0.01)
k_needed = min_k_for_target(0.05, 0.01)
assert empty_prob(k_needed, 0.05) <= 0.01 and empty_prob(k_needed - 1, 0.05) > 0.01
print(f'φ=5% 且要求空结果率 ≤ 1%: k 至少 {k_needed}')
assert k_needed > 80, f'应当大到不实用，得到 {k_needed}'
assert min_k_for_target(0.0, 0.01) is None, 'φ=0 时无论 k 多大都是空'
print(f"{'φ':>7}{'空结果率≤1% 所需 k':>20}")
for ph in [0.5, 0.2, 0.1, 0.05, 0.01]:
    print(f'{ph:>7.2f}{min_k_for_target(ph, 0.01):>20}')
print('✅ 练习 2 通过：φ 越小，后置过滤所需的 k 越不现实')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def empty_prob(k, phi):
    return (1.0 - phi) ** k

def min_k_for_target(phi, target):
    if phi <= 0:
        return None
    if phi >= 1:
        return 1
    # (1-phi)^k <= target  →  k >= log(target)/log(1-phi)
    k = math.ceil(math.log(target) / math.log(1 - phi))
    while empty_prob(k, phi) > target:
        k += 1
    return max(1, k)

assert abs(empty_prob(10, 0.05) - 0.5987) < 1e-3
assert min_k_for_target(0.5, 0.01) == 7
kn = min_k_for_target(0.05, 0.01)
assert empty_prob(kn, 0.05) <= 0.01 and empty_prob(kn - 1, 0.05) > 0.01 and kn > 80
assert min_k_for_target(0.0, 0.01) is None
print('✅ 参考答案 2 通过')
print(f'   φ=1% 时需要 k={min_k_for_target(0.01, 0.01)}——这已经不是「调大 k」能解决的问题。')
print('   所以「向量库不支持前置过滤」不是一个可以绕过的限制，它是一个选型否决项。')

## ✏️ 练习 3：改写的安全检查

改写会改变语义（第 2 节的漂移事故）。实现一个**上线前的自动检查** `rewrite_safety`：

给定一批 (原查询, 改写后) 对，返回被判为不安全的那些，判据三条（命中任一即不安全）：
1. **否定词丢失**：原查询含否定词（不/无/非/未/否）而改写后不含
2. **数字改变**：两边的数字多重集不同
3. **长度塌缩**：改写后的字符数少于原查询的一半

In [ ]:
NEG_WORDS = ('不', '无', '非', '未', '否')

def rewrite_safety(pairs):
    """pairs: [(original, rewritten)]。
    返回 [(original, rewritten, [触发的规则名])]，只含不安全的。
    规则名用 'negation_lost' / 'number_changed' / 'length_collapse'。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
PAIRS = [
    ('不含增值税的发票怎么处理', '增值税的发票怎么处理'),        # 否定丢失
    ('超过 5000 元的合同', '超过 500 元的合同'),                 # 数字改变
    ('餐饮报销的单次上限是多少', '餐饮'),                        # 长度塌缩
    ('吃饭最多能花多少钱', '餐饮单次上限是多少钱'),               # 安全
    ('电脑多久换一次', '笔记本电脑多久换一次'),                   # 安全
    ('不超过 5000 元的合同', '5 元的合同'),                      # 三条全中
]
bad = rewrite_safety(PAIRS)
names = {o: set(rules) for o, _, rules in bad}
print(f'{len(bad)}/{len(PAIRS)} 个改写被判不安全:')
for o, r, rules in bad:
    print(f'  「{o}」 → 「{r}」  {rules}')

assert len(bad) == 4, f'应当有 4 个不安全，得到 {len(bad)}'
assert 'negation_lost' in names['不含增值税的发票怎么处理']
assert 'number_changed' in names['超过 5000 元的合同']
assert 'length_collapse' in names['餐饮报销的单次上限是多少']
assert names['不超过 5000 元的合同'] == {'negation_lost', 'number_changed', 'length_collapse'}
assert '吃饭最多能花多少钱' not in names and '电脑多久换一次' not in names
print('✅ 练习 3 通过：三条规则都能独立触发，也能同时触发')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def rewrite_safety(pairs):
    out = []
    for orig, rew in pairs:
        rules = []
        if any(w in orig for w in NEG_WORDS) and not any(w in rew for w in NEG_WORDS):
            rules.append('negation_lost')
        if Counter(re.findall(r'\d+', orig)) != Counter(re.findall(r'\d+', rew)):
            rules.append('number_changed')
        if len(rew) < len(orig) / 2:
            rules.append('length_collapse')
        if rules:
            out.append((orig, rew, rules))
    return out

bad = rewrite_safety(PAIRS)
names = {o: set(rules) for o, _, rules in bad}
assert len(bad) == 4
assert names['不超过 5000 元的合同'] == {'negation_lost', 'number_changed', 'length_collapse'}
assert '吃饭最多能花多少钱' not in names
print('✅ 参考答案 3 通过')
print('   这三条都是确定性检查（零误报），可以直接阻断改写器上线。')
print('   注意它们不检查「改写是不是变好了」——那需要评测集，属于统计检查。')
print('   确定性检查的作用是拦住**明确的语义破坏**，这是两件不同的事。')

## ✏️ 练习 4：跨表述敏感度报告

讲解第 8 节的三条纪律里，最有信息量的是「每个问题准备多个改述并分别报分」。

实现 `paraphrase_report(groups, k=3)`：`groups` 是 `{gold_chunk_id: [表述1, 表述2, ...]}`。
返回 `dict(per_group, mean_recall, sensitivity, worst)`：
- `per_group[gold]` = 该组的命中率（多少个表述能召回到 gold）
- `mean_recall` = 所有表述的总命中率
- `sensitivity` = **同一问题的不同表述之间命中与否的不一致比例**
  （即 0 < per_group < 1 的组数 / 总组数）
- `worst` = 命中率最低的那个组的 gold id

In [ ]:
def paraphrase_report(groups, k=K_EVAL):
    """返回 dict(per_group, mean_recall, sensitivity, worst)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
GROUPS = {
    'c01': ['餐饮报销的单次上限是多少', '吃饭最多能花多少钱', '餐费能报多少'],
    'c05': ['正式员工每年享有多少天带薪年假', '休假有几天', '正式工年假多少天'],
    'c07': ['笔记本电脑的更换周期是多久', '电脑多久换一次', '笔记本几年一换'],
    'c08': ['密码长度不得少于多少位', '密码要多长', '口令最短几位'],
}
rep = paraphrase_report(GROUPS)
print('每组命中率:', {g: round(v, 2) for g, v in rep['per_group'].items()})
print(f"总命中率 {rep['mean_recall']:.2f} | 表述敏感度 {rep['sensitivity']:.2f} "
      f"| 最差组 {rep['worst']}")

assert set(rep['per_group']) == set(GROUPS)
assert all(0.0 <= v <= 1.0 for v in rep['per_group'].values())
tot = sum(len(v) for v in GROUPS.values())
manual = sum(1 for g, qs in GROUPS.items() for q in qs
             if g in [c for c, _ in retrieve(q, k=K_EVAL)]) / tot
assert abs(rep['mean_recall'] - manual) < 1e-9
assert 0.0 < rep['sensitivity'] <= 1.0, '这些组里应当存在表述敏感的组'
assert rep['worst'] in GROUPS
assert rep['per_group'][rep['worst']] == min(rep['per_group'].values())
print('✅ 练习 4 通过：敏感度比平均 recall 更有信息量——')
print('   它直接告诉你「换个说法就不行了」这件事发生的频率')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def paraphrase_report(groups, k=K_EVAL):
    per_group, hits, total = {}, 0, 0
    for gold, qs in groups.items():
        h = sum(1 for q in qs if gold in [c for c, _ in retrieve(q, k=k)])
        per_group[gold] = h / len(qs)
        hits += h; total += len(qs)
    mixed = sum(1 for v in per_group.values() if 0.0 < v < 1.0)
    worst = min(per_group, key=lambda g: (per_group[g], g))
    return dict(per_group=per_group, mean_recall=hits / total,
                sensitivity=mixed / len(groups), worst=worst)

rep = paraphrase_report(GROUPS)
tot = sum(len(v) for v in GROUPS.values())
manual = sum(1 for g, qs in GROUPS.items() for q in qs
             if g in [c for c, _ in retrieve(q, k=K_EVAL)]) / tot
assert abs(rep['mean_recall'] - manual) < 1e-9
assert 0.0 < rep['sensitivity'] <= 1.0
assert rep['per_group'][rep['worst']] == min(rep['per_group'].values())
print('✅ 参考答案 4 通过')
print(f"   敏感度 {rep['sensitivity']:.0%} 的含义：这个比例的问题「换个说法就变了结果」。")
print('   查询侧的所有改动都应该在这个数上看收益，而不是在平均 recall 上——')
print('   平均 recall 会被大量「本来就能答对」的样本冲淡。')

## 🧪 真实工程胶囊：查询侧的落地

```python
# ══════════════════════════════════════════════════════════════════
# A. 路由：规则优先，偏向检索（讲解第 2 节）
# ══════════════════════════════════════════════════════════════════
def route(query, history):
    if SMALLTALK_RE.match(query):          return 'skip'
    if ARITHMETIC_RE.search(query):        return 'skip'
    if needs_sql(query):                   return 'sql'      # 跨库路由
    return 'retrieve'                                        # 兜底：查
#   上线顺序：先只加 skip 规则并统计命中率，确认漏检索为 0 后再考虑 LLM 路由器。

# ══════════════════════════════════════════════════════════════════
# B. 多轮改写：把指代消解掉（这是改写最大的单项收益）
# ══════════════════════════════════════════════════════════════════
REWRITE_PROMPT = '\n'.join([
    '把用户的最新问题改写成一个不依赖对话历史的独立问题。',
    '只输出改写后的问题，不要解释。',
    '保留原问题里的所有数字、否定词与限定条件。',      # ← 对应练习 3 的三条检查
    '',
    '历史：{history}',
    '最新问题：{q}',
])
#   改写结果必须过 rewrite_safety()（练习 3）才允许使用；不过就退回原查询。

# ══════════════════════════════════════════════════════════════════
# C. 扩展与融合：三路，失败模式互不相同（讲解第 4 节）
# ══════════════════════════════════════════════════════════════════
variants = [q_original, q_rewritten, hyde(q_original)]
rankings = await asyncio.gather(*[vsearch(v, k=k * 2, where=where) for v in variants])
final = rrf(rankings, k0=60, k=k)
#   容量提醒：m=3 意味着向量库按 3 倍 QPS 做规划（讲解第 7 节）。

# ══════════════════════════════════════════════════════════════════
# D. 过滤必须前置，且租户不可绕过（讲解第 5 节）
# ══════════════════════════════════════════════════════════════════
def vsearch(q, k, where=None, *, tenant):        # tenant 是关键字必填参数
    where = {'$and': [{'tenant': tenant}, where]} if where else {'tenant': tenant}
    res = col.query(query_embeddings=[enc(q)], n_results=k, where=where)
    if not res['ids'][0]:
        alert('empty_candidate_set', q=q, where=where)        # 报警，不静默
    return res
#   把 tenant 做成必填关键字参数，而不是可选参数——
#   可选参数把一次跨租户泄漏的距离缩短到一行漏写的代码。

# ══════════════════════════════════════════════════════════════════
# E. 缓存（讲解第 7 节）
# ══════════════════════════════════════════════════════════════════
key = sha256(q, ROUTE_VERSION, REWRITER_ID, sha256(REWRITE_PROMPT),
             SYNONYM_VERSION, MODEL_ID, TEMPERATURE, M_EXPAND)
#   温度 > 0 时缓存「一整组」改写，不缓存单次结果。

# ══════════════════════════════════════════════════════════════════
# F. 评测集（讲解第 8 节）—— 这一项决定了前面五项能不能被正确评估
# ══════════════════════════════════════════════════════════════════
#   1) 查询来自真实日志或「没看过文档的人」
#   2) 每题 2-3 个改述，分别报分，报 sensitivity（练习 4）
#   3) 单独维护一个「跨词汇」子集，查询侧改动只在它上面看收益
#   4) CI 里跑一次 word_overlap 自检，重叠率显著高于线上就说明评测集偏乐观
```

---

## 小结

| 结论 | 数字 | 在哪一节 |
|---|---|---|
| 四个动作的顺序不能换；③ 与 ④ 不可交换 | 过滤必须作用在检索之前 | 讲解 1 |
| 「一律检索」对不该检索的查询是净损害 | 正确率从 100% 掉下来 | 第 1 节 |
| 路由器的目标是「漏检索为 0」，不是总准确率 | 两类错误代价不对称 | 第 1 节 |
| 任何改写都可能改变语义 | 「不含增值税」→「增值税」 | 第 2 节 |
| HyDE 的伪答案不需要正确 | 相似度 0.05 → 0.48（连数字写错也一样） | 第 3 节 |
| 融合降的是方差，不保证提高均值 | 融合 0.83 vs 最好单路 1.00、最差 0.33 | 第 4 节 |
| 后置过滤会静默返回空 | $(1-\phi)^k$；φ=5%、k=10 → 60% | 第 5 节 |
| 向量检索对否定几乎失明 | 语义相反句对 cos 0.70–0.94 | 第 6 节 |
| 改写 prompt 的哈希必须进缓存键 | 漏了就「改了没效果」 | 第 7 节 |
| 用文档原句当查询会掩盖查询侧的全部价值 | 在该集上改写收益为 0 | 第 8 节 |

下一模块：**04 · 迭代与图检索**——多跳问题的单轮 recall 天然是 0，
而迭代必须带停止准则，否则成本没有上界。